# 20 — Reflection, time direction and observable motion

This notebook makes the laterality study's transformation laws executable.
We use **synthetic trajectories throughout**. An observable is a declared
numerical function of measured movement. Equivariance specifies how that
value changes under a transformation; it does not guarantee the value is
present in a learned representation.

Reflection M swaps anatomical left/right landmarks and negates centered
horizontal coordinates. It moves confidence and validity with the joints.
Reversal T reverses the trajectory and its sequence of physical time intervals.
We will test M² = T² = identity and MT = TM, then show why a speed target
cannot by itself establish that a model understands temporal order.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display, SVG, Markdown
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/gavd6_sjepa').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
EVIDENCE = ROOT / 'work/artifacts/iclr-bridge-2026-09-11'
PANEL = ROOT / 'outputs/iclr-bridge-cached-20260911'
def read_json(path):
    return json.loads(Path(path).read_text())
pd.set_option('display.precision', 8)
print('Repository:', ROOT)
print('Execution: read-only evidence inspection + labeled synthetic calculations')

In [ ]:
display(SVG(filename=str(ROOT / 'docs/studies/iclr/figures/03_reflection_and_time.svg')))

## 1. Run the fixed calibration

The fixture's seed, source allocation and tolerances are declared in its
module before computation. Its algebra checks use a 10⁻¹² tolerance.
The source-held prediction example uses 48 generated sources, each with
two opposite-order trajectories; 36 sources train and 12 test. This is a
controlled example with known structure, not a sample of real people.

Calling `run_calibration()` without an output path has no filesystem writes.
Every result below is freshly calculated. The saved JSON in the audit folder
records the same fixture together with predictions and fit identities.

In [ ]:
from gavd6_sjepa.research_directions.iclr_bridge.symmetry_calibration import run_calibration
calibration = run_calibration()
assert calibration['status'] == 'passed'
display(pd.DataFrame([calibration['group_errors']]))
display(pd.DataFrame(calibration['observable_parities']))

## 2. Interpret the four transformation classes

Total speed stays unchanged under reflection and reversal. The bilateral
speed contrast changes sign under reflection but is unchanged by reversal.
Signed vertical displacement changes under reversal; averaging the two
sides makes it reflection-even, while contrasting sides makes it
reflection-odd. Here “vertical” is the image-coordinate y direction,
not a calibrated gravity vector.

Speeds use only valid adjacent observations. With an occluded gap, subtracting
the last visible coordinate from a later coordinate would be a different
measurement. Reversal must reverse unequal time intervals too; otherwise
it changes speed estimates and invalidates the intended parity test.

Biological dynamics need not be reversible. T is a mathematical diagnostic
on a bounded observed sequence. It does not justify putting future frames
into a deployed prefix encoder or treating reverse videos as valid forecasts.

## 3. Exact geometry can coexist with no information

For a feature function h in a shared coordinate basis, its parity component
is P_ab h(x) = [h(x) + a h(Mx) + b h(Tx) + ab h(MTx)] / 4, with signs
a and b each ±1. This ordinary finite-group projection enforces the desired
sign laws. It is not a novel learning theorem. A constant zero feature
satisfies every odd transformation constraint perfectly.

Thus a symmetry penalty or exact architecture needs two separate checks:
whether its output transforms correctly, and whether that output predicts
a nontrivial independent observable. Energy or rank alone cannot replace
the second check; random features can have both.

In [ ]:
display(pd.DataFrame(calibration['projector_checks']))
display(pd.DataFrame([calibration['zero_feature_control']]))

## 4. Create an ambiguity that genuinely requires history

Each generated source supplies a path and its reversal. The paths end at
the same current posture and have the same distribution of coordinates,
confidence and support. Their last velocities have opposite signs. We
define a synthetic one-step continuation using that last velocity. This
continuation rule is part of the fixture, not an assumption about humans.

Means, standard deviations and absolute adjacent changes are identical
within each pair. A model using only those summaries cannot tell the two
futures apart. Ordered signed velocities can. This explains why the
laterality expanded-summary gain cannot be called proof of temporal-order
learning: several features it adds are invariant to reversal.

In [ ]:
import matplotlib.pyplot as plt
temporal = calibration['temporal_observability']
wave = np.asarray(temporal['illustrative_wave'])
times = np.asarray(temporal['illustrative_times'])
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(times, wave, '-o', label='Generated history A', color='#087F8C')
ax.plot(times, wave[::-1], '-s', label='Reversed history B', color='#7953A5')
ax.set(xlabel='Synthetic time (seconds)', ylabel='Synthetic y coordinate',
       title='Same endpoint; opposite final velocity')
ax.legend(); ax.grid(alpha=.2); plt.show()
display(pd.DataFrame([{k:r[k] for k in ('features','r2_training_mean_reference','nominal_features','supported_features')}
                      for r in temporal['scores']]))
assert set(temporal['train_sources']).isdisjoint(temporal['test_sources'])

## 5. Transfer the lesson to the real experiment without transferring the result

The fixture shows that the selected readout can recover a declared signal
and that order-even summaries miss it by construction. The cached GAVD
panel uses its existing 32-frame, four-bin representation and its designated
four-frame block shuffle, not this eleven-frame teaching fixture. Its
production-path planted-history test separately exercises the actual finite
search. Neither calibration predicts the sign of the real-data result.

A useful real extension will measure both the existing reflection-odd,
time-even contrast and a time-odd future observable. It will match current
posture and support, include motion-direction controls, and test an actual
student's held-source readout. For a video teacher, paired reflection
components require encoding two real videos in a common feature basis.
Existing pooled vectors cannot be “mirrored” by swapping skeleton columns.

Continue to **21** to inspect the new real cached comparison and to **22**
for the target-selection and distillation design.